In [22]:
# ============================================================
# SETUP
# ============================================================
get_ipython().system('pip install earthengine-api geemap -q')

import ee
ee.Authenticate()
ee.Initialize(project='osmgee')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import itertools
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

In [3]:
import re
from datetime import datetime

asset_folder_rema = 'projects/annual-ccdc-new/assets/REMA_ONLY_PlanetImages'

assets_info_rema = ee.data.listAssets({'parent': asset_folder_rema})
asset_ids_rema = [a['id'] for a in assets_info_rema['assets']]



In [4]:
# Group into {prefix: {'image': ..., 'udm': ...}} pairs
rema_pairs = {}
for asset_id in asset_ids_rema:
    name = asset_id.split('/')[-1]
    match = re.match(r'(\d{8}_\d{6}_\d+_[\da-zA-Z]+)', name)
    if not match:
        print(f'WARNING: could not parse prefix from {name}')
        continue
    prefix = match.group(1)

    if 'AnalyticMS_SR' in name:
        rema_pairs.setdefault(prefix, {})['image'] = asset_id
    elif 'udm2' in name:
        rema_pairs.setdefault(prefix, {})['udm'] = asset_id



In [6]:
# Sanity check: every prefix should have BOTH an image and a udm
incomplete = {k: v for k, v in rema_pairs.items() if 'image' not in v or 'udm' not in v}
if incomplete:
    print('WARNING: incomplete pairs found:', incomplete)
else:
    print(f'All {len(rema_pairs)} scenes have matched image+mask pairs.')

# Extract the acquisition date from each prefix (first 8 characters, YYYYMMDD)
for prefix in rema_pairs:
    date_str = prefix[:8]
    rema_pairs[prefix]['date'] = datetime.strptime(date_str, '%Y%m%d').strftime('%Y-%m-%d')

# Print a clean summary
for prefix, info in sorted(rema_pairs.items()):
    print(info['date'], '->', prefix)

All 20 scenes have matched image+mask pairs.
2021-01-07 -> 20210107_043441_12_2407
2021-01-07 -> 20210107_043443_39_2407
2021-01-31 -> 20210131_043531_17_2403
2021-01-31 -> 20210131_043533_46_2403
2021-01-31 -> 20210131_043535_75_2403
2021-02-12 -> 20210212_043349_52_2401
2021-02-12 -> 20210212_043351_79_2401
2021-02-12 -> 20210212_043354_06_2401
2021-02-22 -> 20210222_043514_36_2416
2021-02-22 -> 20210222_043516_72_2416
2021-03-04 -> 20210304_043333_93_2413
2021-03-04 -> 20210304_043336_19_2413
2021-03-04 -> 20210304_043338_46_2413
2021-03-17 -> 20210317_034258_42_2206
2021-03-17 -> 20210317_034300_63_2206
2021-03-25 -> 20210325_034756_62_225a
2021-03-25 -> 20210325_034758_90_225a
2021-03-25 -> 20210325_034801_18_225a
2021-03-25 -> 20210325_043553_13_2407
2021-03-25 -> 20210325_043555_40_2407


In [7]:
def mask_planet_clouds_shadows(analytic_image, udm_image):
    """Same logic as your JS: keep pixels where cloud (b6) and shadow (b3) are both 0."""
    cloud_mask = udm_image.select('b6').eq(0)
    shadow_mask = udm_image.select('b3').eq(0)
    clear_mask = cloud_mask.And(shadow_mask)
    return analytic_image.updateMask(clear_mask)



In [8]:
# Build the masked, dated image list automatically from rema_pairs
rema_masked_images = []

for prefix, info in rema_pairs.items():
    analytic_img = ee.Image(info['image'])
    udm_img = ee.Image(info['udm'])

    masked = mask_planet_clouds_shadows(analytic_img, udm_img)
    masked = masked.set('system:time_start', ee.Date(info['date']).millis())
    masked = masked.set('scene_id', prefix)

    rema_masked_images.append(masked)

print(f'Built {len(rema_masked_images)} masked images')

# Assemble into an ImageCollection, same as your JS ee.ImageCollection.fromImages([...])
rema_planet_collection = ee.ImageCollection.fromImages(rema_masked_images)

print('Rema-Kalenga Planet collection size:', rema_planet_collection.size().getInfo())
print('Date range:', 
      rema_planet_collection.aggregate_min('system:time_start').getInfo(),
      'to',
      rema_planet_collection.aggregate_max('system:time_start').getInfo())

Built 20 masked images
Rema-Kalenga Planet collection size: 20
Date range: 1609977600000 to 1616630400000


In [14]:
rema_boundary = ee.FeatureCollection('projects/osmgee/assets/RemaKalenga')
knp_boundary = ee.FeatureCollection('projects/osmgee/assets/KNP')

In [13]:
rema_planet_composite = rema_planet_collection.median().clip(rema_boundary)

planet_band_names = ['coastal_blue', 'blue', 'green_i', 'green', 'yellow', 'red', 'red_edge', 'nir']
rema_planet_composite = rema_planet_composite.rename(planet_band_names)

print('Bands in Rema-Kalenga Planet composite:', rema_planet_composite.bandNames().getInfo())

Bands in Rema-Kalenga Planet composite: ['coastal_blue', 'blue', 'green_i', 'green', 'yellow', 'red', 'red_edge', 'nir']


In [15]:
sample_stats = rema_planet_composite.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=rema_boundary.geometry(),
    scale=30,
    maxPixels=1e9,
    bestEffort=True
)
print('Value ranges per band:')
print(sample_stats.getInfo())

Value ranges per band:
{'blue_max': 752, 'blue_min': 246, 'coastal_blue_max': 616, 'coastal_blue_min': 193, 'green_i_max': 1116, 'green_i_min': 429, 'green_max': 1061, 'green_min': 333, 'nir_max': 3416, 'nir_min': 996, 'red_edge_max': 1580, 'red_edge_min': 549, 'red_max': 1298, 'red_min': 233, 'yellow_max': 1489, 'yellow_min': 459}


In [16]:
# Apply the 10,000 scaling factor to convert DN to actual reflectance (0-1)
rema_planet_composite_scaled = rema_planet_composite.divide(10000)

# Verify the scaled range looks correct
scaled_stats = rema_planet_composite_scaled.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=rema_boundary.geometry(),
    scale=30,
    maxPixels=1e9,
    bestEffort=True
)
print('Scaled value ranges per band (should now be roughly 0-1):')
print(scaled_stats.getInfo())

Scaled value ranges per band (should now be roughly 0-1):
{'blue_max': 0.0752, 'blue_min': 0.0246, 'coastal_blue_max': 0.0616, 'coastal_blue_min': 0.0193, 'green_i_max': 0.1116, 'green_i_min': 0.0429, 'green_max': 0.1061, 'green_min': 0.0333, 'nir_max': 0.3416, 'nir_min': 0.0996, 'red_edge_max': 0.158, 'red_edge_min': 0.0549, 'red_max': 0.1298, 'red_min': 0.0233, 'yellow_max': 0.1489, 'yellow_min': 0.0459}


In [17]:
def add_indices_planet(image):
    ndvi = image.normalizedDifference(['nir', 'red']).rename('NDVI')
    gndvi = image.normalizedDifference(['nir', 'green']).rename('GNDVI')

    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('nir'),
            'RED': image.select('red'),
            'BLUE': image.select('blue')
        }).rename('EVI')

    savi = image.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5', {
            'NIR': image.select('nir'),
            'RED': image.select('red')
        }).rename('SAVI')

    msavi = image.expression(
        '(2 * NIR + 1 - sqrt((2 * NIR + 1)**2 - 8 * (NIR - RED))) / 2', {
            'NIR': image.select('nir'),
            'RED': image.select('red')
        }).rename('MSAVI')

    return image.addBands([ndvi, gndvi, evi, savi, msavi])

rema_planet_with_indices = add_indices_planet(rema_planet_composite_scaled)

# Select the 8 raw Planet bands + 5 indices (13 total, vs. 15 for Sentinel-2's 10+5)
planet_raw_bands = ['coastal_blue', 'blue', 'green_i', 'green', 'yellow', 'red', 'red_edge', 'nir']
index_bands = ['NDVI', 'EVI', 'SAVI', 'GNDVI', 'MSAVI']

final_image_rema_planet = rema_planet_with_indices.select(planet_raw_bands + index_bands)

print('Bands in final Rema-Kalenga Planet image:', final_image_rema_planet.bandNames().getInfo())

Bands in final Rema-Kalenga Planet image: ['coastal_blue', 'blue', 'green_i', 'green', 'yellow', 'red', 'red_edge', 'nir', 'NDVI', 'EVI', 'SAVI', 'GNDVI', 'MSAVI']


In [19]:
# Reload Rema-Kalenga sites (same as the Sentinel-2 script)
rema_sites = ee.FeatureCollection('projects/osmgee/assets/RemaSites')

def clean_feature_rema(f):
    lon = ee.Number(f.get('XCoord'))
    lat = ee.Number(f.get('YCoord'))
    par_lai = ee.Number(f.get('MEAN_PAR_L'))
    cluster_id = f.get('ClusterID')

    return ee.Feature(
        ee.Geometry.Point([lon, lat]),
        {
            'cluster_id': cluster_id,
            'par_lai': par_lai,
            'longitude': lon,
            'latitude': lat
        }
    )

rema_sites_clean = rema_sites.map(clean_feature_rema)

print('Cleaned feature count:', rema_sites_clean.size().getInfo())
print('Sample feature:', rema_sites_clean.first().getInfo())

Cleaned feature count: 59
Sample feature: {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [91.62458, 24.11235]}, 'id': '00000000000000000004', 'properties': {'cluster_id': 5, 'latitude': 24.11235, 'longitude': 91.62458, 'par_lai': 0.33117018625}}


In [20]:
# Rebuild the 15m x 15m square plots
def make_square_plot(feature):
    point = feature.geometry()
    half_side = 7.5
    square = point.buffer(half_side, 1).bounds()
    return feature.setGeometry(square)

rema_sites_squares = rema_sites_clean.map(make_square_plot)
print('Sample square geometry:', rema_sites_squares.first().geometry().getInfo())

Sample square geometry: {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[91.6245064407107, 24.112283907026445], [91.62465051076359, 24.112283907026445], [91.62465051076359, 24.112417491494657], [91.6245064407107, 24.112417491494657], [91.6245064407107, 24.112283907026445]]]}


In [23]:
rema_planet_extracted = final_image_rema_planet.reduceRegions(
    collection=rema_sites_squares,
    reducer=ee.Reducer.mean(),
    scale=3
)

rema_planet_extracted_info = rema_planet_extracted.getInfo()

for f in rema_planet_extracted_info['features'][:5]:
    print(f['properties'])
    print('---')

rema_planet_rows = [f['properties'] for f in rema_planet_extracted_info['features']]
df_rema_planet = pd.DataFrame(rema_planet_rows)

id_cols = ['cluster_id', 'par_lai', 'longitude', 'latitude']
other_cols = [c for c in df_rema_planet.columns if c not in id_cols]
df_rema_planet = df_rema_planet[id_cols + other_cols]

print(f'Extracted {len(df_rema_planet)} plots from Planet imagery')
df_rema_planet

{'EVI': 0.3890045058764712, 'GNDVI': 0.6788505965654505, 'MSAVI': 0.34191173472663455, 'NDVI': 0.7138372915277189, 'SAVI': 0.3730095781359969, 'blue': 0.030946819526627214, 'cluster_id': 5, 'coastal_blue': 0.024309556213017756, 'green': 0.0438171449704142, 'green_i': 0.052648949704142, 'latitude': 24.11235, 'longitude': 91.62458, 'nir': 0.22907296597633137, 'par_lai': 0.33117018625, 'red': 0.038259341715976336, 'red_edge': 0.07465868343195267, 'yellow': 0.06388076923076921}
---
{'EVI': 0.38921208988553024, 'GNDVI': 0.6839223994202975, 'MSAVI': 0.33885779043639547, 'NDVI': 0.7382221492740658, 'SAVI': 0.3714832095216512, 'blue': 0.029419434576672593, 'cluster_id': 27, 'coastal_blue': 0.02498015097690941, 'green': 0.04117778271166371, 'green_i': 0.04793423623445828, 'latitude': 24.165805, 'longitude': 91.6306, 'nir': 0.21939702486678508, 'par_lai': 3.94414734825, 'red': 0.03304658081705151, 'red_edge': 0.06827563647128479, 'yellow': 0.05235827412670218}
---
{'EVI': 0.42765543660516175, 'G

,cluster_id,par_lai,longitude,latitude,EVI,GNDVI,MSAVI,NDVI,SAVI,blue,coastal_blue,green,green_i,nir,red,red_edge,yellow
0,5,0.331170,91.624580,24.112350,0.389005,0.678851,0.341912,0.713837,0.373010,0.030947,0.024310,0.043817,0.052649,0.229073,0.038259,0.074659,0.063881
1,27,3.944147,91.630600,24.165805,0.389212,0.683922,0.338858,0.738222,0.371483,0.029419,0.024980,0.041178,0.047934,0.219397,0.033047,0.068276,0.052358
2,35,2.494383,91.643603,24.170542,0.427655,0.701676,0.378806,0.757648,0.405113,0.029053,0.027350,0.042667,0.050808,0.243417,0.033549,0.074569,0.054202
3,51,3.001718,91.625308,24.203328,0.351603,0.637710,0.304154,0.661268,0.338413,0.034735,0.025975,0.047554,0.055366,0.215128,0.043862,0.078823,0.066772
4,3,2.859987,91.632832,24.109208,0.420541,0.692874,0.368622,0.749445,0.396500,0.030549,0.022464,0.043217,0.049600,0.238410,0.034131,0.072928,0.060871
5,4,2.068093,91.626706,24.110476,0.430616,0.684456,0.379209,0.752051,0.405229,0.030703,0.024135,0.046022,0.053603,0.245707,0.034796,0.078836,0.063656
6,7,3.134443,91.629136,24.113164,0.392537,0.696166,0.341434,0.757936,0.374399,0.027835,0.020684,0.038645,0.046815,0.215837,0.029713,0.065186,0.056126
7,8,2.119183,91.630112,24.114924,0.439071,0.709573,0.387829,0.774302,0.412932,0.028671,0.023264,0.041561,0.050972,0.244697,0.031104,0.070365,0.060674
8,9,3.243282,91.631906,24.116332,0.426811,0.705995,0.375734,0.767811,0.402996,0.028645,0.022212,0.040988,0.049427,0.237951,0.031246,0.067666,0.058904
9,10,4.013032,91.635002,24.115710,0.474998,0.725289,0.423777,0.796869,0.441978,0.028246,0.021951,0.041957,0.049551,0.263598,0.029786,0.072127,0.058112


In [24]:
asset_folder_knp = 'projects/annual-ccdc-new/assets/KNP_REMA_PlanetImages'

assets_info_knp = ee.data.listAssets({'parent': asset_folder_knp})
asset_ids_knp = [a['id'] for a in assets_info_knp['assets']]

print(f'Total assets found (KNP folder): {len(asset_ids_knp)}')
for aid in asset_ids_knp:
    print(aid)

Total assets found (KNP folder): 30
projects/annual-ccdc-new/assets/KNP_REMA_PlanetImages/20220102_042552_31_2414_3B_AnalyticMS_SR_8b_harmonized_clip
projects/annual-ccdc-new/assets/KNP_REMA_PlanetImages/20220102_042552_31_2414_3B_udm2_clip
projects/annual-ccdc-new/assets/KNP_REMA_PlanetImages/20220102_042554_61_2414_3B_AnalyticMS_SR_8b_harmonized_clip
projects/annual-ccdc-new/assets/KNP_REMA_PlanetImages/20220102_042554_61_2414_3B_udm2_clip
projects/annual-ccdc-new/assets/KNP_REMA_PlanetImages/20220103_034042_15_242d_3B_AnalyticMS_SR_8b_harmonized_clip
projects/annual-ccdc-new/assets/KNP_REMA_PlanetImages/20220103_034042_15_242d_3B_udm2_clip
projects/annual-ccdc-new/assets/KNP_REMA_PlanetImages/20220105_033632_71_2465_3B_AnalyticMS_SR_8b_harmonized_clip
projects/annual-ccdc-new/assets/KNP_REMA_PlanetImages/20220105_033632_71_2465_3B_udm2_clip
projects/annual-ccdc-new/assets/KNP_REMA_PlanetImages/20220108_042639_10_2414_3B_AnalyticMS_SR_8b_harmonized_clip
projects/annual-ccdc-new/asset

In [25]:
asset_folder_knp = 'projects/annual-ccdc-new/assets/KNP_REMA_PlanetImages'

assets_info_knp = ee.data.listAssets({'parent': asset_folder_knp})
asset_ids_knp = [a['id'] for a in assets_info_knp['assets']]

# Group into {prefix: {'image': ..., 'udm': ...}} pairs
knp_pairs = {}
for asset_id in asset_ids_knp:
    name = asset_id.split('/')[-1]
    match = re.match(r'(\d{8}_\d{6}_\d+_[\da-zA-Z]+)', name)
    if not match:
        print(f'WARNING: could not parse prefix from {name}')
        continue
    prefix = match.group(1)

    if 'AnalyticMS_SR' in name:
        knp_pairs.setdefault(prefix, {})['image'] = asset_id
    elif 'udm2' in name:
        knp_pairs.setdefault(prefix, {})['udm'] = asset_id

incomplete = {k: v for k, v in knp_pairs.items() if 'image' not in v or 'udm' not in v}
if incomplete:
    print('WARNING: incomplete pairs found:', incomplete)
else:
    print(f'All {len(knp_pairs)} scenes have matched image+mask pairs.')

for prefix in knp_pairs:
    date_str = prefix[:8]
    knp_pairs[prefix]['date'] = datetime.strptime(date_str, '%Y%m%d').strftime('%Y-%m-%d')

for prefix, info in sorted(knp_pairs.items()):
    print(info['date'], '->', prefix)

All 15 scenes have matched image+mask pairs.
2022-01-02 -> 20220102_042552_31_2414
2022-01-02 -> 20220102_042554_61_2414
2022-01-03 -> 20220103_034042_15_242d
2022-01-05 -> 20220105_033632_71_2465
2022-01-08 -> 20220108_042639_10_2414
2022-01-09 -> 20220109_033645_24_245c
2022-01-09 -> 20220109_042506_11_227a
2022-01-17 -> 20220117_033650_28_241f
2022-02-07 -> 20220207_033620_57_2423
2022-02-13 -> 20220213_042524_78_2414
2022-02-14 -> 20220214_035507_84_2231
2022-02-15 -> 20220215_033621_76_2460
2022-03-07 -> 20220307_033759_74_2440
2022-03-17 -> 20220317_040552_02_247a
2022-03-21 -> 20220321_040904_17_2492


In [26]:
# Apply masking (same function as Rema-Kalenga — reused, not redefined)
knp_masked_images = []

for prefix, info in knp_pairs.items():
    analytic_img = ee.Image(info['image'])
    udm_img = ee.Image(info['udm'])

    masked = mask_planet_clouds_shadows(analytic_img, udm_img)
    masked = masked.set('system:time_start', ee.Date(info['date']).millis())
    masked = masked.set('scene_id', prefix)

    knp_masked_images.append(masked)

print(f'Built {len(knp_masked_images)} masked images')

knp_planet_collection = ee.ImageCollection.fromImages(knp_masked_images)

print('KNP Planet collection size:', knp_planet_collection.size().getInfo())

Built 15 masked images
KNP Planet collection size: 15


In [27]:
# Reload KNP boundary and sites for this session
knp_boundary = ee.FeatureCollection('projects/osmgee/assets/KNP')
knp_sites = ee.FeatureCollection('projects/osmgee/assets/SitesLocation')

print('KNP boundary features:', knp_boundary.size().getInfo())
print('KNP sites features:', knp_sites.size().getInfo())


KNP boundary features: 1
KNP sites features: 40


In [28]:
# Build the median composite, clipped to KNP boundary
knp_planet_composite = knp_planet_collection.median().clip(knp_boundary)

# Rename bands (same SuperDove order as Rema-Kalenga)
planet_band_names = ['coastal_blue', 'blue', 'green_i', 'green', 'yellow', 'red', 'red_edge', 'nir']
knp_planet_composite = knp_planet_composite.rename(planet_band_names)

print('Bands in KNP Planet composite:', knp_planet_composite.bandNames().getInfo())

# Apply the same 10,000 scaling factor
knp_planet_composite_scaled = knp_planet_composite.divide(10000)

# Verify scaled range
knp_scaled_stats = knp_planet_composite_scaled.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=knp_boundary.geometry(),
    scale=30,
    maxPixels=1e9,
    bestEffort=True
)
print('Scaled value ranges per band (KNP):')
print(knp_scaled_stats.getInfo())

Bands in KNP Planet composite: ['coastal_blue', 'blue', 'green_i', 'green', 'yellow', 'red', 'red_edge', 'nir']
Scaled value ranges per band (KNP):
{'blue_max': 0.0564, 'blue_min': 0.0157, 'coastal_blue_max': 0.0519, 'coastal_blue_min': 0.0197, 'green_i_max': 0.0784, 'green_i_min': 0.0248, 'green_max': 0.0814, 'green_min': 0.0206, 'nir_max': 0.407, 'nir_min': 0.1107, 'red_edge_max': 0.1371, 'red_edge_min': 0.0309, 'red_max': 0.0965, 'red_min': 0.014, 'yellow_max': 0.1013, 'yellow_min': 0.0251}


In [29]:
# Apply the same index function (already defined for Planet bands)
knp_planet_with_indices = add_indices_planet(knp_planet_composite_scaled)

planet_raw_bands = ['coastal_blue', 'blue', 'green_i', 'green', 'yellow', 'red', 'red_edge', 'nir']
index_bands = ['NDVI', 'EVI', 'SAVI', 'GNDVI', 'MSAVI']

final_image_knp_planet = knp_planet_with_indices.select(planet_raw_bands + index_bands)

print('Bands in final KNP Planet image:', final_image_knp_planet.bandNames().getInfo())

Bands in final KNP Planet image: ['coastal_blue', 'blue', 'green_i', 'green', 'yellow', 'red', 'red_edge', 'nir', 'NDVI', 'EVI', 'SAVI', 'GNDVI', 'MSAVI']


In [30]:
# Rebuild KNP site geometries in this session (fresh session, same fix as before)
def clean_feature_knp(f):
    lon = ee.Number(f.get('XCoord'))
    lat = ee.Number(f.get('YCoord'))
    par_lai = ee.Number(f.get('MEAN_PAR_L'))
    cluster_id = f.get('ClusterID')

    return ee.Feature(
        ee.Geometry.Point([lon, lat]),
        {
            'cluster_id': cluster_id,
            'par_lai': par_lai,
            'longitude': lon,
            'latitude': lat
        }
    )

knp_sites_clean = knp_sites.map(clean_feature_knp)

def make_square_plot(feature):
    point = feature.geometry()
    half_side = 7.5
    square = point.buffer(half_side, 1).bounds()
    return feature.setGeometry(square)

knp_sites_squares = knp_sites_clean.map(make_square_plot)

print('Cleaned KNP feature count:', knp_sites_clean.size().getInfo())

Cleaned KNP feature count: 40


In [31]:
# Extract mean values over each 15x15m square plot from the Planet composite
knp_planet_extracted = final_image_knp_planet.reduceRegions(
    collection=knp_sites_squares,
    reducer=ee.Reducer.mean(),
    scale=3
)

knp_planet_extracted_info = knp_planet_extracted.getInfo()

knp_planet_rows = [f['properties'] for f in knp_planet_extracted_info['features']]
df_knp_planet = pd.DataFrame(knp_planet_rows)

id_cols = ['cluster_id', 'par_lai', 'longitude', 'latitude']
other_cols = [c for c in df_knp_planet.columns if c not in id_cols]
df_knp_planet = df_knp_planet[id_cols + other_cols]

print(f'Extracted {len(df_knp_planet)} plots from KNP Planet imagery')
df_knp_planet

Extracted 40 plots from KNP Planet imagery


,cluster_id,par_lai,longitude,latitude,EVI,GNDVI,MSAVI,NDVI,SAVI,blue,coastal_blue,green,green_i,nir,red,red_edge,yellow
0,20,1.670617,91.942780,24.952030,0.520846,0.770549,0.480870,0.849334,0.485984,0.022305,0.027670,0.036996,0.039047,0.285333,0.023254,0.070833,0.039605
1,2,3.038662,91.937182,24.952476,0.411997,0.751492,0.367764,0.830497,0.397370,0.020045,0.027559,0.030525,0.033141,0.215908,0.019833,0.054387,0.034447
2,27,2.154577,91.943748,24.945720,0.569630,0.741819,0.530833,0.797189,0.523175,0.030024,0.032969,0.051857,0.050760,0.350094,0.039529,0.103017,0.056374
3,40,0.285474,91.961718,24.971386,0.449598,0.715166,0.409375,0.757515,0.429420,0.027793,0.032571,0.044408,0.044737,0.267433,0.036957,0.085460,0.050971
4,11,3.923973,91.938690,24.952073,0.449765,0.758807,0.408448,0.839741,0.430497,0.019980,0.027198,0.032683,0.034705,0.239570,0.020710,0.059893,0.035831
5,13,3.148044,91.938552,24.954740,0.418524,0.747953,0.374982,0.813667,0.403699,0.021348,0.027087,0.032280,0.035335,0.224280,0.022950,0.062292,0.037595
6,25,4.923975,91.941830,24.948150,0.321227,0.724939,0.277961,0.798572,0.320158,0.018521,0.025638,0.026155,0.030991,0.164498,0.018321,0.044827,0.031520
7,32,1.538075,91.958823,24.968853,0.391722,0.750634,0.346706,0.812152,0.380415,0.020806,0.027402,0.029306,0.033749,0.206210,0.021352,0.053999,0.035778
8,33,3.215866,91.959881,24.966343,0.542770,0.790886,0.508478,0.857394,0.505903,0.021185,0.027414,0.035202,0.036346,0.301837,0.023172,0.073273,0.038672
9,36,3.479967,91.960536,24.968643,0.359190,0.734161,0.316254,0.798483,0.353198,0.019829,0.027112,0.028778,0.031290,0.189130,0.020932,0.051965,0.034448
